In [2]:
import random as rnd
import numpy as np

In [3]:
# class that implements tableau and necessary operations
# Reference -Improved Simulation of Stabilizer Circuits by Scott Aaronson & Daniel Gottesman
class Cirq_Tableau:
    
    def __init__(
        self,
        pauli_word: list[str] = None  
    ):
        if pauli_word is None or len(pauli_word) == 0:
            self._column_num = None
            self._row_num = None

            self._ss, self._xs, self._zs = None, None, None
        else:
            if "-" in pauli_word[0]:
                self._column_num = len(pauli_word[0][1:]) 
            else:
                self._column_num = len(pauli_word[0])
            self._row_num = len(pauli_word)

            self._ss = self.create_sign(pauli_word)
            self._xs, self._zs = self.create_tab(pauli_word)

    # setters & getters 
    @property
    def ss(self) -> np.ndarray:
        return self._ss

    @ss.setter 
    def ss(self, new_ss: np.ndarray):
        self._ss = new_ss
        
    @property
    def xs(self) -> np.ndarray:
        return self._xs

    @xs.setter 
    def xs(self, new_xs: np.ndarray):
        self._xs = new_xs
        
    @property
    def zs(self) -> np.ndarray:
        return self._zs

    @zs.setter 
    def zs(self, new_zs: np.ndarray):
        self._zs = new_zs
        
    @property
    def column_num(self) -> int:
        return self._column_num

    @column_num.setter 
    def column_num(self, new_num: int):
        self._column_num = new_num
    
    @property
    def row_num(self) -> int:
        return self._row_num

    @row_num.setter 
    def row_num(self, new_num: int):
        self._row_num = new_num

    # functions that create parts of tableau
    def create_sign(self, pauli_word: list[str]):
        temp_ss = np.zeros((self.row_num), dtype=int)
        for pauli_string in pauli_word:
            if "-" in pauli_string:
                index = pauli_word.index(pauli_string)
                pauli_word[index] = pauli_string.replace("-", "")
                #print(pauli_word)
                temp_ss[index] = 1
        return temp_ss
        
    def create_tab(self, pauli_word: list[str]):
        temp_x = np.zeros((self.row_num, self.column_num), dtype=int)
        temp_z = np.zeros((self.row_num, self.column_num), dtype=int)
        for i, pauli_string in enumerate(pauli_word):
            for j, pauli in enumerate(pauli_string):
                if pauli == "X" or pauli == "Y":
                    temp_x[i][j] = 1
                if pauli == "Z" or pauli == "Y":
                    temp_z[i][j] = 1
        return temp_x, temp_z

    # Clifford operations on tableau. 
    def apply_H(self,column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.xs[:, column], self.zs[:, column] = self.zs[:, column].copy(), self.xs[:, column].copy()

    def apply_S(self, column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.zs[:, column] = self.xs[:, column] ^ self.zs[:, column]

    def apply_CX(self, control: int, target: int):
        self.ss ^= (
            (self.xs[:, control] & self.zs[:, target])
            &(~(self.xs[:, target] ^ self.zs[:, control]))
        )
        self.xs[:, target] ^= self.xs[:, control]
        self.zs[:, control] ^= self.zs[:, target]

    # class operations necessary for comparisons and equating
    def copy(self):
        new_tab = Cirq_Tableau()
        new_tab.column_num = self.column_num
        new_tab.row_num = self.row_num
        new_tab.ss = self.ss.copy()
        new_tab.zs = self.zs.copy()
        new_tab.xs = self.xs.copy()
        return new_tab
    
    def __eq__(self, other):
        if not isinstance(other, type(self)):
            return NotImplemented  
        return (
            self.column_num == other.column_num
            and self.row_num == other.row_num
            and np.array_equal(self.ss, other.ss)
            and np.array_equal(self.xs, other.xs)
            and np.array_equal(self.zs, other.zs)
        )
    
    def return_string(self):
        string = ''
        for i in range(self.row_num):
            if self.ss[i]:
                string += "-"  

            for j in range(self.column_num):
                if self.xs[i][j] and not self.zs[i][j]:
                    string += "X"
                elif not self.xs[i][j] and self.zs[i][j]:
                    string += "Z"
                elif self.xs[i][j] and self.zs[i][j]:
                    string += "Y"
                else:
                    string += "I"
            if i < self.row_num - 1:
                string += "\n" 

        return string
    
    def __copy__(self):
        return self.copy()
        

    def __str__(self) -> str:
        ss = np.expand_dims(self.ss, axis = 1)
        xz = np.concatenate((self.xs, self.zs, ss), axis=1)
        return str(xz)

    def __hash__(self) -> int:
        return hash(self.zs.tobytes() + self.xs.tobytes() + self.ss.tobytes())

    def __lt__(self, other):
        return self.row_num < other.row_num

In [102]:
def reduce(tab, ndx, ctrl, targ, path):
    '''
    Function to reduce a single pair of non-identity Paulis

    ndx:int index of current Pauli word being acted on
    ctrl:int control qubit
    targ:int target qubit
    path:list[tuples] list of current applied actions
    '''

    # large condition 
    if tab.xs[ndx][ctrl] & ~tab.zs[ndx][ctrl]: # if control is X
        if tab.xs[ndx][targ] & ~tab.zs[ndx][targ]:  # if target is X
            operations = [[("CX", ctrl, targ, 1)], [("S", ctrl, 0),("CX", ctrl, targ, 1)], 
                              [("H", ctrl, 0), ("H", targ, 0), ("CX", ctrl, targ, 1)], 
                          [("H", ctrl, 0), ("S", targ, 0), ("CX", ctrl, targ, 1)]]
            op = rnd.choice(operations)
            path += op
            for i in op:
                match i[0]:
                    case "CX":
                        tab.apply_CX(i[1], i[2])
                    case "S":
                        tab.apply_S(i[1])
                    case "H":
                        tab.apply_H(i[1])
                            
        elif tab.xs[ndx][targ] & tab.zs[ndx][targ]: # if target is Y
            operations = [[("S", ctrl, 0),("S", targ, 0),("CX", ctrl, targ, 1)], [("S", targ, 0),("CX", ctrl, targ, 1)],
                             [("H", ctrl, 0),("CX", ctrl, targ, 1)], [("H", ctrl, 0), ("H", targ, 0), ("CX", ctrl, targ, 1)]]
            op = rnd.choice(operations)
            path += op
            for i in op:
                match i[0]:
                    case "CX":
                        tab.apply_CX(i[1], i[2])
                    case "S":
                        tab.apply_S(i[1])
                    case "H":
                        tab.apply_H(i[1])
                            
        elif ~tab.xs[ndx][targ] & tab.zs[ndx][targ]: # if target is Y
            operations = [[("H", ctrl, 0),("S", targ, 0),("CX", ctrl, targ, 1)], [("S", ctrl, 0),("H", targ, 0), ("CX", ctrl, targ, 1)],
                             [("H", ctrl, 0),("CX", ctrl, targ, 1)], [("H", targ, 0),("CX", ctrl, targ, 1)]]
            op = rnd.choice(operations)
            path += op
            for i in op:
                match i[0]:
                    case "CX":
                        tab.apply_CX(i[1], i[2])
                    case "S":
                        tab.apply_S(i[1])
                    case "H":
                        tab.apply_H(i[1])
                            
    elif tab.xs[ndx][ctrl] & tab.zs[ndx][ctrl]: # if control is Y
        if tab.xs[ndx][targ] & ~tab.zs[ndx][targ]:  # if target is X
            operations = [[("CX", ctrl, targ, 1)], [("S", ctrl, 0),("CX", ctrl, targ, 1)], 
                              [("H", ctrl, 0), ("CX", ctrl, targ, 1)], ]
            op = rnd.choice(operations)
            path += op
            for i in op:
                match i[0]:
                    case "CX":
                        tab.apply_CX(i[1], i[2])
                    case "S":
                        tab.apply_S(i[1])
                    case "H":
                        tab.apply_H(i[1])
                        
        elif tab.xs[ndx][targ] & tab.zs[ndx][targ]:  # if target is Y
            operations = [[("S", ctrl, 0),("S", targ, 0),("CX", ctrl, targ, 1)], [("S", targ, 0),("CX", ctrl, targ, 1)],
                             [("H", ctrl, 0),("S", targ, 0),("CX", ctrl, targ, 1)]]
            op = rnd.choice(operations)
            path += op
            for i in op:
                match i[0]:
                    case "CX":
                        tab.apply_CX(i[1], i[2])
                    case "S":
                        tab.apply_S(i[1])
                    case "H":
                        tab.apply_H(i[1])
                            
        elif ~tab.xs[ndx][targ] & tab.zs[ndx][targ]:  # if target is Z
            operations = [ [("S", ctrl, 0),("H", targ, 0), ("CX", ctrl, targ, 1)], [("H", targ, 0),("CX", ctrl, targ, 1)]]
            op = rnd.choice(operations)
            path += op
            for i in op:
                match i[0]:
                    case "CX":
                        tab.apply_CX(i[1], i[2])
                    case "S":
                        tab.apply_S(i[1])
                    case "H":
                        tab.apply_H(i[1])
                        
    elif ~tab.xs[ndx][ctrl] & tab.zs[ndx][ctrl]: # if control is Z
        if tab.xs[ndx][targ] & ~tab.zs[ndx][targ]:  # if target is X
            operations = [[("S", ctrl, 0),("H", targ, 0),("CX", ctrl, targ, 1)], 
                              [("H", ctrl, 0), ("CX", ctrl, targ, 1)], [("S", ctrl, 0),("S", targ, 0),("CX", ctrl, targ, 1)]]
            op = rnd.choice(operations)
            path += op
            for i in op:
                match i[0]:
                    case "CX":
                        tab.apply_CX(i[1], i[2])
                    case "S":
                        tab.apply_S(i[1])
                    case "H":
                        tab.apply_H(i[1])
                        
        elif tab.xs[ndx][targ] & tab.zs[ndx][targ]:  # if target is Y
            operations = [[("CX", ctrl, targ, 1)], [("S", ctrl, 0),("CX", ctrl, targ, 1)], [("H", targ, 0),("CX", ctrl, targ, 1)],
                             [("S", ctrl, 0),("H", targ, 0),("CX", ctrl, targ, 1)]]
            op = rnd.choice(operations)
            path += op
            for i in op:
                match i[0]:
                    case "CX":
                        tab.apply_CX(i[1], i[2])
                    case "S":
                        tab.apply_S(i[1])
                    case "H":
                        tab.apply_H(i[1])
                            
        elif ~tab.xs[ndx][targ] & tab.zs[ndx][targ]:  # if target is Z
            operations = [ [("CX", ctrl, targ, 1)], [("S", ctrl, 0),("S", targ, 0), ("CX", ctrl, targ, 1)], 
                          [("S", targ, 0),("CX", ctrl, targ, 1)], [("S", ctrl, 0),("CX", ctrl, targ, 1)], 
                          [("H", ctrl, 0),("H", targ, 0), ("CX", ctrl, targ, 1)]]
            op = rnd.choice(operations)
            path += op
            for i in op:
                match i[0]:
                    case "CX":
                        tab.apply_CX(i[1], i[2])
                    case "S":
                        tab.apply_S(i[1])
                    case "H":
                        tab.apply_H(i[1])

In [103]:
def rnd_solve(tableau, ndx):
    '''
    Function to create a random solution for 1 Pauli string in a Pauli word

    tab: Cirq_tableau that holds Pauli word to be acted on
    ndx: int that holds the index of the Pauli string we want to solve
    '''
    path = [] # list to store path of operations that reduces the Pauli string
    tab = tableau.copy() # copies tableau to not alter it

    # loop to run process until solution is found
    while True:

        # array that holds 1 where a non-identity is located 
        weight_string = tab.xs[ndx, :] | tab.zs[ndx, :]

        # condition to break loop
        if sum(weight_string) == 1:
            return tab, path

        # all indexes for non-identity Paulis
        positions = np.where(weight_string == 1)[0].tolist()

        # random selection of non-identity indexes
        ndxs = rnd.sample(positions, 2)
        ctrl = ndxs[0]
        targ = ndxs[1]

        # function that provides 1 block that can reduce the pair provided
        reduce(tab, ndx, ctrl, targ, path)
            

In [108]:
word = Cirq_Tableau(["XZXZXZIIX","ZYZZIYZYZ", "YYZYXIXXZ", "ZYIXXZZZZ", "YXYZIXXIY", "YXZZYXYIX" ])
tb, path = rnd_solve(word, 3)
print(tb.return_string())
print(path)

XXXZIXIII
XYZIXYIYY
IZZYXXYXZ
-IIIIIIIIZ
-XYYIXZYZZ
-YIZIZZZZX
[('H', 6, 0), ('CX', 1, 6, 1), ('S', 7, 0), ('CX', 7, 0, 1), ('H', 1, 0), ('CX', 1, 4, 1), ('CX', 0, 1, 1), ('S', 3, 0), ('H', 5, 0), ('CX', 3, 5, 1), ('S', 1, 0), ('S', 3, 0), ('CX', 1, 3, 1), ('H', 1, 0), ('CX', 1, 8, 1)]
